In [ ]:
fig_interactive = jwspecfit.plot_fit_interactive(result)
fig_interactive.show()

## Interactive plotly plot

Zoom in on any line by clicking and dragging.  Hover to see the
flux of each component at any wavelength.

In [ ]:
jwspecfit.export_lines_txt(result, "prism_lines.txt")

# Preview the file
with open("prism_lines.txt") as f:
    print(f.read())

## Export line measurements to text

Save a table with flux, EW, centroid, velocity width, integrated SNR,
and peak SNR for each line.

In [ ]:
# Save
jwspecfit.save_result(result, "prism_fit_result.npz")

# Reload (no need to re-fit)
loaded = jwspecfit.load_result("prism_fit_result.npz")
print(f"Loaded: {len(loaded.lines)} lines, χ²/dof = {loaded.chi2:.2f}")

fig = jwspecfit.plot_fit(loaded)
plt.show()

## Save and reload a fit result

Save the full fit (model, parameters, line measurements) to a `.npz`
file, then reload it later for replotting without re-running the fit.

In [ ]:
# Fit only the Hα region (rest ~6563 Å → observed ~46000 Å at z=6)
result_window = jwspecfit.fit_lines(
    spec, z=6.0,
    wave_range_A=(40000, 50000),
    n_boot=0,
)
fig = jwspecfit.plot_fit(result_window)
plt.show()

# 01 — Fitting a Prism Spectrum

This notebook demonstrates the basic `jwspecfit` workflow:
1. Load a JWST NIRSpec PRISM spectrum from FITS
2. Fit all observable emission lines (with bootstrap uncertainties by default)
3. Inspect per-line results (flux, SNR, EW)
4. Plot the fit with Gaussian components (in Angstroms)
5. Save/load fit results and export line measurements
6. Interactive plotly plot for zooming into line profiles
7. Fit a restricted wavelength window

The prism has wavelength-dependent resolution R(λ) ≈ 30–300, which
`jwspecfit` handles automatically from the FITS header.

In [ ]:
import jwspecfit
import matplotlib.pyplot as plt

print(f"jwspecfit v{jwspecfit.__version__}")

## Load the spectrum

The FITS file has a `SPEC1D` HDU with columns `wave` (µm), `flux` (µJy),
and `err` (µJy).  The grating is read from the header automatically.

In [ ]:
spec = jwspecfit.read_fits("../../data/borg-v4_prism-clear_1747_732.spec.fits", z=6.0)

print(f"Grating:    {spec.grating}")
print(f"Pixels:     {spec.n_pix}")
print(f"Wave range: {spec.wave_um.min():.3f} – {spec.wave_um.max():.3f} µm")

## Quick look at the raw spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
valid = spec.mask_valid()
ax.step(spec.wave_um[valid], spec.flux_ujy[valid], where="mid", lw=0.8, color="0.3")
ax.fill_between(
    spec.wave_um[valid],
    (spec.flux_ujy - spec.err_ujy)[valid],
    (spec.flux_ujy + spec.err_ujy)[valid],
    step="mid", alpha=0.15, color="0.5",
)
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Flux density [$\mu$Jy]")
ax.set_title("Raw PRISM spectrum")
plt.tight_layout()

## Fit emission lines

A single call does everything: continuum subtraction, line detection,
Gaussian fitting, and bootstrap uncertainties (200 iterations by default).
The grating and resolution are auto-detected from the FITS header.

In [ ]:
result = jwspecfit.fit_lines(spec, z=6.0)

print(f"Fit success: {result.success}")
print(f"χ²/dof:      {result.chi2:.2f}")
print(f"Lines:       {len(result.lines)}")

## Per-line results

Each line has flux, bootstrap uncertainty, SNR, equivalent width,
centroid, and Gaussian width.

In [ ]:
print(f"{'Line':<18s} {'Flux':>12s} {'Flux err':>12s} {'SNR':>8s} {'EW (Å)':>10s} {'σ (Å)':>8s}")
print("-" * 72)
for name, lr in result.lines.items():
    print(
        f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e} {lr.snr:8.1f} {lr.ew_A:10.1f} {lr.sigma_A:8.1f}"
    )

## Plot the fit

The plot shows data, continuum, total model, and individual Gaussian
components as filled curves.  The y-axis is scaled to the tallest
emission line so the lines are clearly visible.

In [ ]:
fig = jwspecfit.plot_fit(result)
plt.show()

## Quick fit (analytic errors)

For fast exploration, set `n_boot=0` to skip bootstrap and use analytic
error estimates instead.

In [ ]:
result_fast = jwspecfit.fit_lines(spec, z=6.0, n_boot=0)

print(f"{'Line':<18s} {'Flux':>12s} {'Err (analytic)':>14s} {'SNR':>8s}")
print("-" * 56)
for name, lr in result_fast.lines.items():
    if lr.snr > 2:
        print(f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:14.3e} {lr.snr:8.1f}")

## Fit specific lines only

You can restrict the fit to a subset of lines:

In [ ]:
result_oiii = jwspecfit.fit_lines(
    spec, z=6.0,
    lines=["OIII_4959", "OIII_5007", "HBETA"],
)
fig = jwspecfit.plot_fit(result_oiii)
plt.show()

## Fit a wavelength window

Restrict the continuum and line fitting to a specific wavelength range
(in Angstroms). Useful for focusing on a single line complex.